# AICE Professional 주피터라이트 실전 모의고사 11회 — 공개데이터(서울 따릉이 대여량 · 영화 리뷰 감정(NSMC) · 콩잎 병해)

## [ 필독 ] AICE Professional 모의고사 유의사항

- 공식 샘플문항(가공식품 카테고리 분류)과 **같은 시험지 형식**으로 만든 모의고사입니다. 3문항, 180분 기준입니다.
- 공식 시험과 같은 **Tabular / Text / Image 3유형**입니다. 2번 Text 문항은 공식 샘플(상품명 → 카테고리)과 같은 텍스트 분류입니다.
- 본인 스스로 모델을 만들고 **달성 목표 이상의 성능**이 나오도록 개발해야 합니다. 정답(테스트 라벨)은 제공되지 않습니다.
  대신 이 노트북 **맨 마지막 셀**(전체 자동 채점, 정답 키 인코딩)로 제출 파일을 스스로 채점할 수 있습니다(연습용).

## 사용할 수 있는 라이브러리 (주피터라이트 판)

이 모의고사는 **브라우저 주피터라이트(Pyodide · Python 3.14)** 에서 설치 없이 끝까지 풀 수 있도록 라이브러리를 제한했습니다.
필요한 라이브러리는 주피터라이트에 **미리 설치**되어 있습니다. 맨 위 '브라우저 실습' 셀은 확인만 하고 바로 끝납니다(Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다).

| 구분 | 쓸 수 있음 | 쓰지 않음(브라우저에서 안 됨) |
|---|---|---|
| 데이터·그림 | numpy · pandas · matplotlib · seaborn · Pillow | — |
| 머신러닝 | scikit-learn · LightGBM · XGBoost · imbalanced-learn · joblib | CatBoost |
| 텍스트 | scikit-learn 의 CountVectorizer · TfidfVectorizer · 내장 영어 불용어, **NLTK**(불용어·토큰화·어간 추출, 데이터는 `/drive/nltk_data` 에 미리 들어 있음) | 형태소 분석기 · BERT 등 사전학습 언어 모델 |
| 딥러닝 | `tensorflow.keras` 문법(TensorFlow.js 로 실행되는 대체 구현체, 미리 설치): `Sequential`·함수형(`Input` → 층 → `Model`) · Dense · Dropout · BatchNormalization · Conv2D · MaxPooling2D · GlobalAveragePooling2D · Embedding · LSTM · GRU · 콜백 3종(EarlyStopping · ModelCheckpoint · ReduceLROnPlateau) · `load_img` · `img_to_array` · `image_dataset_from_directory` · `ImageDataGenerator` · `model.save` / `load_model` · **MobileNet v1 전이학습**(`alpha=0.25`, 224×224, `weights='imagenet'` 만) | PyTorch · 다른 사전학습 모델(MobileNetV2/V3 · ResNet · EfficientNet 등) · 사용자 정의 층·손실 · GPU |

- 이미지는 64×64 정도로 줄여 작은 CNN 을 직접 학습하거나, MobileNet v1(alpha 0.25)로 특징을 뽑아 분류기를 학습하세요(전이학습). 브라우저 TensorFlow.js 는 CPython TensorFlow 와 수치가 조금 달라 딥러닝 점수가 약간 다르게 나올 수 있습니다.
- 학습이 오래 걸리면 epoch 를 줄이거나 조기 종료(EarlyStopping)를 쓰세요. 최신 Chrome·Edge 를 권장합니다.

## 시간 배분(180분)

| 문항 | 권장 시간 | 먼저 할 일 |
|---|---|---|
| 1. Tabular (30점) | 50분 | `info()`·`describe()` 로 자료형·결측 확인 → 기본 모델로 제출 파일부터 만들기 |
| 2. Text (35점) | 60분 | 라벨 분포·문장 길이 확인 → TF-IDF + 선형 모델로 기준 점수 |
| 3. Image (35점) | 60분 | 압축 풀기·폴더 구조 확인 → 작은 CNN 으로 기준 점수 |
| 마무리 | 10분 | 제출 파일 6개 이름·열 구조 확인 → 맨 마지막 채점 셀 실행 |

## 시험 자원 가이드

| 구분 | 공식 시험 | 이 모의고사 |
|---|---|---|
| IDE | CPU 8 core · Memory 32G · GPU 1개(VRAM 16GB) | **브라우저 주피터라이트**(설치 없음, CPU) — Colab·로컬 Jupyter 에서도 그대로 실행됨 |
| SW | Python 3.10 · TensorFlow 2.13 · PyTorch 2.1 | Python 3.14(Pyodide) · scikit-learn · LightGBM · XGBoost · Keras 문법(TensorFlow.js) — 위 표의 라이브러리만 |

## 유의 사항

- 막히면 문항 아래 접혀 있는 **힌트 1·2·3단계**를 순서대로 펼쳐 보세요. 단계가 올라갈수록 정답에 가까워집니다.
- 제출 파일명은 **고정**입니다: `answer_1~3` 모델 파일과 `answer_1~3.csv`.


문제에 제시되는 가이드는 필수적으로 준수하여 주시기 바랍니다. 이를 준수하지 않을 경우 채점 제한 또는 0점 처리될 수 있으니
각별히 유의해 주시기 바랍니다.

#### **[학습 정보]**
- 학습 목표 : Tabular · Text · Image 예측 모델 3종을 만들어 달성 목표 이상의 예측 파일을 제출하고, 제출한 코드로 같은 모델이 재현되게 할 수 있다
- AICE 역량 : PT1·PT2 표 데이터 / PX1·PX2 텍스트 / PI1·PI2 이미지 / PS1 제출 파일 형식·재현성
- 제출 파일 : `answer_1~3` 모델 파일(.pkl · .h5 등)과 `answer_1~3.csv` — **파일명은 고정**입니다(이름을 바꾸면 채점되지 않음)

| 문항 | 문제 | 달성 목표 | 역량 | 기술 | 예상 시간 |
|---|---|---|---|---|---|
| 1. Tabular | 서울 공공자전거(따릉이) 시간대별 대여량 예측 | RMSE 152.6 이하 | PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | pandas, scikit-learn(RandomForestRegressor, KFold), LightGBM, joblib | 50분 |
| 2. Text | 영화 리뷰 감정 분류(긍정·부정) | 정확도 0.77 이상 | PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | pandas, scikit-learn(CountVectorizer, TfidfVectorizer, MultinomialNB, LinearSVC), joblib | 60분 |
| 3. Image | 콩잎 사진으로 병해 진단(3종) | 정확도 0.77 이상 | PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | pandas, scikit-learn(LogisticRegression), Keras(TensorFlow.js), joblib | 60분 |

In [1]:
# (브라우저 실습) 주피터라이트에는 tensorflow(TensorFlow.js 대체 구현체)·scikit-learn·LightGBM 등이 미리 설치되어 있어 이 셀은 확인만 합니다.
# Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다.
import os, sys, importlib.util
if 'pyodide' in sys.modules:
    os.environ['AICE_IMG_DIR'] = '/tmp/aice_images/' + os.path.basename(os.getcwd())   # 이미지 압축은 메모리 폴더에 풂(빠름)
    if not ('tensorflow' in sys.modules or importlib.util.find_spec('tensorflow')):
        # 미리 설치되지 않은 사이트(패치 전 주피터라이트)에서만: 같은 폴더의 tfjs_keras.py 를 불러옵니다
        sys.path[:0] = [os.getcwd(), '/drive']
        import tfjs_keras
        await tfjs_keras.load()
print('준비 완료 — 다음 셀로 넘어가세요.')

준비 완료 — 다음 셀로 넘어가세요.


## 1. Tabular 문제 - 서울 공공자전거(따릉이) 시간대별 대여량 예측 (30점)

<sub>학습 목표: 서울 공공자전거(따릉이) 시간대별 대여량 예측 모델로 테스트 RMSE 152.6 이하을 달성할 수 있다 | 역량: PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | 기술: pandas, scikit-learn(RandomForestRegressor, KFold), LightGBM, joblib | 난이도: 실전 | 예상 시간: 50분</sub>

### 문제 개요

서울 공공자전거(따릉이)의 **시간대별 대여 대수** 를 날짜·시각·기상(기온·습도·풍속·강수 등)·계절·휴일 정보로 예측하는 회귀 문제입니다. 출퇴근 시각과 날씨에 따라 수요가 크게 달라지므로, 날짜에서 **월·요일** 같은 특징을 만들어 쓰는 것이 핵심입니다.

제공된 훈련 데이터셋을 활용해 회귀 모델을 개발하고, 테스트 데이터셋의 'rented' 값을 예측하시기 바랍니다.

### 달성 목표

"01_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **RMSE 152.6 이하**를 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `01_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `01_test_x.csv`: 테스트 데이터셋입니다. 라벨('rented') 값이 제외되어 있습니다.

**Label 정보**

- 'rented' : 해당 시각의 대여 대수

**Data 정보**

- 'id' : 행 번호
- 'date' : 날짜(일/월/년 문자열, 예: 01/12/2017)
- 'hour' : 시각(0~23)
- 'temp_c · humidity · wind_ms · visibility_10m · dew_point_c' : 기온(℃) · 습도(%) · 풍속(m/s) · 가시거리(10m) · 이슬점(℃)
- 'solar_mj · rain_mm · snow_cm' : 일사량(MJ/m²) · 강수량(mm) · 적설(cm)
- 'season · holiday · functioning' : 계절 · 공휴일 여부 · 운영일 여부(문자열)

**데이터 출처** — Seoul Bike Sharing Demand(UCI 560 · CC BY 4.0, 서울시 공공데이터 기반) — 2017-12 ~ 2018-11 시간대별 대여량 8,760건

### 데이터 예시

`01_train.csv`

| id | date | hour | temp_c | humidity | wind_ms | visibility_10m | dew_point_c | … |
|---|---|---|---|---|---|---|---|---|
| 1 | 15/03/2018 | 6 | 14.4 | 96 | 0.7 | 453 | 13.7 | … |
| 2 | 10/08/2018 | 20 | 32.2 | 52 | 2.3 | 2000 | 21 | … |
| 3 | 23/06/2018 | 0 | 22.7 | 36 | 1.5 | 2000 | 6.8 | … |

`01_test_x.csv`

| id | date | hour | temp_c | humidity | wind_ms | visibility_10m | dew_point_c | … |
|---|---|---|---|---|---|---|---|---|
| 1 | 27/04/2018 | 13 | 21 | 36 | 1 | 1164 | 5.3 | … |
| 2 | 30/05/2018 | 8 | 19.3 | 83 | 0.6 | 394 | 16.3 | … |
| 3 | 19/10/2018 | 12 | 17.4 | 33 | 1.1 | 2000 | 0.9 | … |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 서울 공공자전거(따릉이) 시간대별 대여량 예측 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_1.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('rented') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_1.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'rented' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'rented' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- `date` 는 문자열입니다. `pd.to_datetime(format='%d/%m/%Y')` 로 바꿔 **월·요일** 특징을 만드세요.
- season·holiday·functioning 은 문자열 범주라 원-핫 인코딩합니다. 운영하지 않은 날(functioning=No)은 대여량이 0 입니다.
- 평가 지표는 **RMSE** 입니다. 예측값이 음수가 되지 않게 0 아래는 0 으로 자르세요.

### 채점 진행 방식 안내

- 1번 문항 배점 : 30점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_1.csv 로 제출된 예측 데이터를 정답지와 대조하여, **RMSE 167.86 이하** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [2]:

import pandas as pd

train_1 = pd.read_csv("01_train.csv")

print("데이터 크기:", train_1.shape)
display(train_1.head())

train_1.info()

print("결측치:")
print(train_1.isna().sum())

print("중복 행:", train_1.duplicated().sum())

print("대여량 통계:")
display(train_1["rented"].describe())

print("대여량 0인 행:", (train_1["rented"] == 0).sum())

for col in ["season", "holiday", "functioning"]:
    print(col, train_1[col].value_counts().to_dict())


데이터 크기: (7008, 15)


,id,date,hour,temp_c,humidity,wind_ms,visibility_10m,dew_point_c,solar_mj,rain_mm,snow_cm,season,holiday,functioning,rented
0,1,15/03/2018,6,14.4,96,0.7,453,13.7,0.00,8.0,0.0,Spring,No Holiday,Yes,10
1,2,10/08/2018,20,32.2,52,2.3,2000,21.0,0.02,0.0,0.0,Summer,No Holiday,Yes,1702
2,3,23/06/2018,0,22.7,36,1.5,2000,6.8,0.00,0.0,0.0,Summer,No Holiday,Yes,1322
3,4,02/09/2018,13,28.5,42,1.7,1990,14.3,2.94,0.0,0.0,Autumn,No Holiday,Yes,1254
4,5,24/01/2018,5,-15.6,29,3.7,2000,-29.6,0.00,0.0,0.9,Winter,No Holiday,Yes,34


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7008 entries, 0 to 7007
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              7008 non-null   int64  
 1   date            7008 non-null   object 
 2   hour            7008 non-null   int64  
 3   temp_c          7008 non-null   float64
 4   humidity        7008 non-null   int64  
 5   wind_ms         7008 non-null   float64
 6   visibility_10m  7008 non-null   int64  
 7   dew_point_c     7008 non-null   float64
 8   solar_mj        7008 non-null   float64
 9   rain_mm         7008 non-null   float64
 10  snow_cm         7008 non-null   float64
 11  season          7008 non-null   object 
 12  holiday         7008 non-null   object 
 13  functioning     7008 non-null   object 
 14  rented          7008 non-null   int64  
dtypes: float64(6), int64(5), object(4)
memory usage: 821.4+ KB
결측치:
id                0
date              0
hour              0

,rented
count,7008.000000
mean,715.336758
std,651.587574
min,0.000000
25%,193.000000
50%,520.000000
75%,1077.000000
max,3556.000000


대여량 0인 행: 228
season {'Summer': 1790, 'Spring': 1768, 'Autumn': 1750, 'Winter': 1700}
holiday {'No Holiday': 6661, 'Holiday': 347}
functioning {'Yes': 6780, 'No': 228}


In [3]:

from sklearn.model_selection import train_test_split

def make_features_1(df):
    X = df.drop(columns=["id", "rented"], errors="ignore").copy()

    dates = pd.to_datetime(X.pop("date"), format="%d/%m/%Y")
    X["month"] = dates.dt.month
    X["weekday"] = dates.dt.weekday

    X = pd.get_dummies(
        X,
        columns=["season", "holiday", "functioning"],
        dtype=int
    )
    return X

X_1 = make_features_1(train_1)
y_1 = train_1["rented"]

X_train_1, X_valid_1, y_train_1, y_valid_1 = train_test_split(
    X_1, y_1,
    test_size=0.2,
    random_state=42
)

print("전처리 후 변수 개수:", X_1.shape[1])
print("학습 데이터:", X_train_1.shape)
print("검증 데이터:", X_valid_1.shape)
print("학습 정답:", y_train_1.shape)
print("검증 정답:", y_valid_1.shape)

display(X_train_1.head(3))


전처리 후 변수 개수: 19
학습 데이터: (5606, 19)
검증 데이터: (1402, 19)
학습 정답: (5606,)
검증 정답: (1402,)


,hour,temp_c,humidity,wind_ms,visibility_10m,dew_point_c,solar_mj,rain_mm,snow_cm,month,weekday,season_Autumn,season_Spring,season_Summer,season_Winter,holiday_Holiday,holiday_No Holiday,functioning_No,functioning_Yes
1741,13,15.8,36,1.5,1888,0.7,2.59,0.0,0.0,10,4,1,0,0,0,0,1,0,1
3105,2,-5.0,65,0.4,990,-10.5,0.00,0.0,0.2,12,0,0,0,0,1,0,1,0,1
1869,23,-0.4,32,0.6,2000,-15.1,0.00,0.0,0.0,1,6,0,0,0,1,0,1,0,1


In [4]:

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error

models_1 = {
    "Ridge": make_pipeline(
        StandardScaler(),
        Ridge(alpha=10)
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=150,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=1
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
        verbosity=-1,
        n_jobs=1
    )
}

cv_1 = KFold(n_splits=4, shuffle=True, random_state=42)

results_1 = []

for name, model in models_1.items():
    scores = -cross_val_score(
        model,
        X_train_1,
        y_train_1,
        cv=cv_1,
        scoring="neg_root_mean_squared_error",
        n_jobs=1
    )

    model.fit(X_train_1, y_train_1)
    pred = np.clip(model.predict(X_valid_1), 0, None)

    valid_rmse = np.sqrt(
        mean_squared_error(y_valid_1, pred)
    )

    results_1.append({
        "모델": name,
        "CV RMSE": scores.mean(),
        "CV 표준편차": scores.std(),
        "독립 검증 RMSE": valid_rmse
    })

result_df_1 = pd.DataFrame(results_1)
display(result_df_1.sort_values("CV RMSE").round(2))


,모델,CV RMSE,CV 표준편차,독립 검증 RMSE
2,LightGBM,153.53,2.57,148.06
1,RandomForest,181.80,1.26,178.42
0,Ridge,437.25,1.37,430.39


In [5]:

X_date_1 = X_1.copy()

dates_1 = pd.to_datetime(
    train_1["date"], format="%d/%m/%Y"
)

X_date_1["day_index"] = (
    dates_1 - dates_1.min()
).dt.days

X_train_date_1 = X_date_1.loc[X_train_1.index]
X_valid_date_1 = X_date_1.loc[X_valid_1.index]

lgb_date_1 = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1,
    n_jobs=1
)

scores_date_1 = -cross_val_score(
    lgb_date_1,
    X_train_date_1,
    y_train_1,
    cv=cv_1,
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)

lgb_date_1.fit(X_train_date_1, y_train_1)

pred_date_1 = np.clip(
    lgb_date_1.predict(X_valid_date_1),
    0, None
)

rmse_date_1 = np.sqrt(
    mean_squared_error(y_valid_1, pred_date_1)
)

comparison_1 = pd.DataFrame([
    {
        "모델": "기존 LightGBM",
        "CV RMSE": 153.53,
        "CV 표준편차": 2.57,
        "독립 검증 RMSE": 148.06
    },
    {
        "모델": "날짜 특성 추가",
        "CV RMSE": scores_date_1.mean(),
        "CV 표준편차": scores_date_1.std(),
        "독립 검증 RMSE": rmse_date_1
    }
])

display(comparison_1.round(2))


,모델,CV RMSE,CV 표준편차,독립 검증 RMSE
0,기존 LightGBM,153.53,2.57,148.06
1,날짜 특성 추가,146.06,3.63,141.90


In [6]:

import os
import joblib

candidate_path_1 = "answer_1_candidate.pkl"

if os.path.exists(candidate_path_1):
    raise FileExistsError("기존 후보 파일이 있어 덮어쓰지 않습니다.")

lgb_final_1 = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1,
    n_jobs=1
)

lgb_final_1.fit(X_date_1, y_1)

joblib.dump(lgb_final_1, candidate_path_1)

file_size_1 = os.path.getsize(candidate_path_1)

print("학습 행 수:", lgb_final_1.n_features_in_, "개 변수")
print("전체 학습 데이터:", len(y_1), "행")
print("저장 파일:", candidate_path_1)
print("파일 크기:", file_size_1, "bytes")
print("저장 크기 정상:", file_size_1 > 0)


학습 행 수: 20 개 변수
전체 학습 데이터: 7008 행
저장 파일: answer_1_candidate.pkl
파일 크기: 818811 bytes
저장 크기 정상: True


In [8]:

import shutil

loaded_model_1 = joblib.load("answer_1_candidate.pkl")

pred_before_1 = lgb_final_1.predict(X_date_1)
pred_after_1 = loaded_model_1.predict(X_date_1)

same_model_1 = np.allclose(
    pred_before_1, pred_after_1,
    rtol=0, atol=1e-12
)

print("모델 재불러오기 예측 일치:", same_model_1)
assert same_model_1

test_file_1 = (
    "01_test_x.csv"
    if os.path.exists("01_test_x.csv")
    else "01_test_x(1).csv"
)

test_1 = pd.read_csv(test_file_1)

X_test_1 = make_features_1(test_1)

test_dates_1 = pd.to_datetime(
    test_1["date"], format="%d/%m/%Y"
)
train_start_1 = pd.to_datetime(
    train_1["date"], format="%d/%m/%Y"
).min()

X_test_1["day_index"] = (
    test_dates_1 - train_start_1
).dt.days

assert set(X_test_1.columns) == set(X_date_1.columns)
X_test_1 = X_test_1[X_date_1.columns]

pred_test_1 = np.clip(
    loaded_model_1.predict(X_test_1),
    0, None
)

submit_1 = test_1.copy()
submit_1["rented"] = pred_test_1

assert len(submit_1) == 1752
assert list(submit_1.columns) == list(test_1.columns) + ["rented"]
assert submit_1["rented"].notna().all()
assert np.isfinite(submit_1["rented"]).all()
assert (submit_1["rented"] >= 0).all()

if os.path.exists("answer_1.pkl") or os.path.exists("answer_1.csv"):
    raise FileExistsError("기존 제출 파일이 있어 덮어쓰지 않습니다.")

shutil.copy2("answer_1_candidate.pkl", "answer_1.pkl")
submit_1.to_csv("answer_1.csv", index=False)

if not os.path.exists("01_test_x.csv"):
    shutil.copy2(test_file_1, "01_test_x.csv")

check_1 = pd.read_csv("answer_1.csv")

assert check_1.shape == submit_1.shape
assert check_1["id"].equals(test_1["id"])
assert np.allclose(check_1["rented"], pred_test_1)

print("테스트 데이터:", test_1.shape)
print("제출 CSV:", check_1.shape)
print("CSV 예측 일치:", True)
print("모델 파일 크기:", os.path.getsize("answer_1.pkl"))
print("CSV 파일 크기:", os.path.getsize("answer_1.csv"))
display(check_1.head())


모델 재불러오기 예측 일치: True
테스트 데이터: (1752, 14)
제출 CSV: (1752, 15)
CSV 예측 일치: True
모델 파일 크기: 818811
CSV 파일 크기: 160712


,id,date,hour,temp_c,humidity,wind_ms,visibility_10m,dew_point_c,solar_mj,rain_mm,snow_cm,season,holiday,functioning,rented
0,1,27/04/2018,13,21.0,36,1.0,1164,5.3,2.99,0.0,0.0,Spring,No Holiday,Yes,1166.001365
1,2,30/05/2018,8,19.3,83,0.6,394,16.3,0.23,0.0,0.0,Spring,No Holiday,Yes,1738.744483
2,3,19/10/2018,12,17.4,33,1.1,2000,0.9,2.39,0.0,0.0,Autumn,No Holiday,Yes,1166.201028
3,4,25/11/2018,12,6.9,59,0.9,669,-0.5,1.31,0.0,2.2,Autumn,No Holiday,Yes,617.032885
4,5,16/06/2018,4,16.9,63,1.0,2000,9.8,0.00,0.0,0.0,Summer,No Holiday,Yes,297.772673


<details><summary>힌트 1단계 (접근 방향) - 1번 문항</summary>

자료형·결측·정답 분포를 확인합니다. 번호·날짜 원문·누수 열은 빼고 범주형은 원-핫 인코딩합니다. '참고'의 파생 변수를 만든 뒤 선형 모델과 트리 모델(LightGBM 등)을 **교차검증 RMSE** 로 비교해 더 나은 쪽으로 전체 학습하세요.

- 이 문항의 핵심: `date` 는 문자열입니다. `pd.to_datetime(format='%d/%m/%Y')` 로 바꿔 **월·요일** 특징을 만드세요.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 1번 문항</summary>

사용할 도구: `pandas`, `joblib`, `RandomForestRegressor`, `LGBMRegressor`, `KFold`, `cross_val_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html>
- 참고 문서: <https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMRegressor.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 1번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
from sklearn.ensemble import ____
from lightgbm import ____
from sklearn.model_selection import ____, ____

train = pd.read_csv('01_train.csv')
test = pd.read_csv('01_test_x.csv')
train.info(verbose=False)
LABEL = '____'

def feats(df):
    d = df.drop(columns=['____', LABEL], errors='____').copy()
    dt = pd.to_datetime(d.pop('____'), format='____')
    d['____'], d['____'] = dt.dt.month, dt.dt.weekday     # 날짜 → 월·요일
    return pd.get_dummies(d, columns=['____', '____', '____'], dtype=int)

# ▷ 후보 모델을 교차검증으로 비교(점수 출력)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_1.pkl`(모델) · `answer_1.csv`(예측)

</details>

## 2. Text 문제 - 영화 리뷰 감정 분류(긍정·부정) (35점)

<sub>학습 목표: 영화 리뷰 감정 분류(긍정·부정) 모델로 테스트 정확도 0.77 이상을 달성할 수 있다 | 역량: PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | 기술: pandas, scikit-learn(CountVectorizer, TfidfVectorizer, MultinomialNB, LinearSVC), joblib | 난이도: 실전 | 예상 시간: 60분</sub>

### 문제 개요

영화 리뷰 한 줄(평균 약 35자)로 그 리뷰가 **긍정인지 부정인지** 를 분류하는 한국어 감정 분석 문제입니다. 맞춤법이 틀리거나 신조어·초성(ㅋㅋ, ㅠㅠ)이 많아 단어 단위보다 **문자 n-gram** 이 잘 맞습니다.

제공된 훈련 데이터셋을 활용해 텍스트 이진 분류(감정) 모델을 개발하고, 테스트 데이터셋의 'sentiment' 값을 예측하시기 바랍니다.

### 달성 목표

"02_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 77% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `02_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `02_test_x.csv`: 테스트 데이터셋입니다. 라벨('sentiment') 값이 제외되어 있습니다.

**Label 정보**

- 'sentiment' : positive(긍정) / negative(부정)

**Data 정보**

- 'review' : 리뷰 문장
- 'id' : 테스트 행 번호(테스트에만)

**데이터 출처** — NSMC 네이버 영화 리뷰 감정 말뭉치(github e9t/nsmc · CC0) — 긍정·부정 각 5,000건 표본

### 데이터 예시

`02_train.csv`

| review | sentiment |
|---|---|
| 애썼다, 마구잡이 복수극으로 불러주마 | negative |
| 풋풋하고 그립고 가슴아프고 아쉽고 다행이였던 영화 | positive |
| 흥행 完전 실패..-_-;; | negative |

`02_test_x.csv`

| id | review |
|---|---|
| 1 | 로보캅 처럼 재미가 시대에 뒤 떨어짐. |
| 2 | 칼, 도끼, 쇠뭉둥이 들고 싸우고 살인까지 하는 고딩들... 아무리 영화라지만 너무 오바한듯 |
| 3 | 구스 반 산트 최악의 영화임. 너무나 스타일리쉬해보이려는 노력이 처절해보일 정도. Alex를 제외한 조연들의 연기는 재앙의 수준.. <밀크>,<굿윌헌팅>의 나레티브도 <일루펀트>의 아름다움도 없고 오직 허세만 남은 영화 |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 영화 리뷰 감정 분류(긍정·부정) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_2.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('sentiment') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_2.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'sentiment' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'sentiment' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 한국어는 조사·어미가 붙어 같은 단어도 모양이 여러 가지입니다. `TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 3))` 을 써 보세요.
- 긍정·부정이 반씩이라 평가 지표는 **정확도(Accuracy)** 입니다. 테스트 행 순서(`id`)를 유지하고, 라벨 문자열(positive/negative)로 제출하세요.

### 채점 진행 방식 안내

- 2번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_2.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 69.3% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [10]:

import pandas as pd

train_2 = pd.read_csv("02_train.csv")
test_2 = pd.read_csv("02_test_x.csv")

print("학습 데이터:", train_2.shape)
print("테스트 데이터:", test_2.shape)

display(train_2.head())

print("자료형:")
train_2.info()

print("결측치:")
print(train_2.isna().sum())

print("중복 리뷰:", train_2["review"].duplicated().sum())
print("빈 리뷰:", train_2["review"].str.strip().eq("").sum())

print("감정별 데이터 수:")
print(train_2["sentiment"].value_counts())

print("감정별 비율(%):")
print((train_2["sentiment"].value_counts(normalize=True) * 100).round(2))

print("리뷰 글자 수 통계:")
display(train_2["review"].str.len().describe())


학습 데이터: (8000, 2)
테스트 데이터: (2000, 2)


,review,sentiment
0,"애썼다, 마구잡이 복수극으로 불러주마",negative
1,풋풋하고 그립고 가슴아프고 아쉽고 다행이였던 영화,positive
2,흥행 完전 실패..-_-;;,negative
3,절대 절대 보지마세요.,negative
4,레아 세이두를 본 것에 의의를...,negative


자료형:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     8000 non-null   object
 1   sentiment  8000 non-null   object
dtypes: object(2)
memory usage: 125.1+ KB
결측치:
review       0
sentiment    0
dtype: int64
중복 리뷰: 0
빈 리뷰: 0
감정별 데이터 수:
sentiment
negative    4000
positive    4000
Name: count, dtype: int64
감정별 비율(%):
sentiment
negative    50.0
positive    50.0
Name: proportion, dtype: float64
리뷰 글자 수 통계:


,review
count,8000.000000
mean,37.259625
std,30.241950
min,5.000000
25%,17.000000
50%,28.000000
75%,44.000000
max,158.000000


In [11]:

from sklearn.model_selection import train_test_split

X_2 = train_2["review"].str.strip()
y_2 = train_2["sentiment"]

X_train_2, X_valid_2, y_train_2, y_valid_2 = train_test_split(
    X_2,
    y_2,
    test_size=0.2,
    random_state=42,
    stratify=y_2
)

print("학습 데이터:", X_train_2.shape)
print("검증 데이터:", X_valid_2.shape)

print("학습 감정 분포:")
print(y_train_2.value_counts())

print("검증 감정 분포:")
print(y_valid_2.value_counts())

print("학습 리뷰 예시:")
display(X_train_2.head())


학습 데이터: (6400,)
검증 데이터: (1600,)
학습 감정 분포:
sentiment
positive    3200
negative    3200
Name: count, dtype: int64
검증 감정 분포:
sentiment
positive    800
negative    800
Name: count, dtype: int64
학습 리뷰 예시:


,review
5110,다시 한번 느끼지만 김수현 연기 정말 잘하는 배우구나 ... 아이유도 연기 많이 늘...
7389,솔직히 개십노잼이다.. 진짜 있어보일려고 발정난영화다진짜 사람들눈이삐엇나?? 이게재...
1030,잼씀 내면을다느낄수있었음
3237,내가 영화를 잘몰라서 그러는데 이런걸 컬트 영화라고 하는건가?
5152,할리우드 cg랑 우리나라랑 비교하면 어떡하니걔네는 우리나라가 전쟁하고 황폐화되었을때...


In [12]:

import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score

models_2 = {
    "Word + LinearSVC": make_pipeline(
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        ),
        LinearSVC(random_state=42)
    ),
    "Char + LinearSVC": make_pipeline(
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(1, 3),
            min_df=2,
            sublinear_tf=True
        ),
        LinearSVC(random_state=42)
    ),
    "Char + LogisticRegression": make_pipeline(
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(1, 3),
            min_df=2,
            sublinear_tf=True
        ),
        LogisticRegression(max_iter=1000)
    ),
    "Char + MultinomialNB": make_pipeline(
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(1, 3),
            min_df=2,
            sublinear_tf=True
        ),
        MultinomialNB(alpha=0.5)
    )
}

cv_2 = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

results_2 = []

for name, model in models_2.items():
    scores = cross_val_score(
        model,
        X_train_2,
        y_train_2,
        cv=cv_2,
        scoring="accuracy",
        n_jobs=1
    )

    model.fit(X_train_2, y_train_2)
    pred = model.predict(X_valid_2)

    results_2.append({
        "모델": name,
        "CV Accuracy": scores.mean() * 100,
        "CV 표준편차": scores.std() * 100,
        "독립 검증 Accuracy": accuracy_score(
            y_valid_2, pred
        ) * 100
    })

result_df_2 = pd.DataFrame(results_2)

display(
    result_df_2.sort_values(
        "CV Accuracy", ascending=False
    ).round(2)
)


,모델,CV Accuracy,CV 표준편차,독립 검증 Accuracy
3,Char + MultinomialNB,82.69,1.39,81.88
1,Char + LinearSVC,81.62,0.58,81.12
2,Char + LogisticRegression,81.39,1.23,82.06
0,Word + LinearSVC,70.72,0.96,72.19


In [13]:

import os
import joblib
from sklearn.base import clone

candidate_path_2 = "answer_2_candidate.pkl"

if os.path.exists(candidate_path_2):
    raise FileExistsError("기존 후보 파일이 있어 덮어쓰지 않습니다.")

model_final_2 = clone(models_2["Char + MultinomialNB"])

model_final_2.fit(X_2, y_2)

joblib.dump(model_final_2, candidate_path_2)

file_size_2 = os.path.getsize(candidate_path_2)

print("전체 학습 리뷰:", len(X_2))
print("TF-IDF 특성 수:",
      len(model_final_2.named_steps["tfidfvectorizer"].vocabulary_))
print("모델 클래스:",
      model_final_2.named_steps["multinomialnb"].classes_)
print("저장 파일:", candidate_path_2)
print("파일 크기:", file_size_2, "bytes")
print("저장 크기 정상:", file_size_2 > 0)


전체 학습 리뷰: 8000
TF-IDF 특성 수: 39538
모델 클래스: ['negative' 'positive']
저장 파일: answer_2_candidate.pkl
파일 크기: 2085169 bytes
저장 크기 정상: True


In [14]:

import shutil

loaded_model_2 = joblib.load("answer_2_candidate.pkl")

pred_before_2 = model_final_2.predict(X_2)
pred_after_2 = loaded_model_2.predict(X_2)

same_model_2 = np.array_equal(
    pred_before_2, pred_after_2
)

print("모델 재불러오기 예측 일치:", same_model_2)
assert same_model_2

assert test_2["review"].notna().all()

X_test_2 = test_2["review"].str.strip()
pred_test_2 = loaded_model_2.predict(X_test_2)

submit_2 = test_2.copy()
submit_2["sentiment"] = pred_test_2

assert len(submit_2) == 2000
assert list(submit_2.columns) == ["id", "review", "sentiment"]
assert submit_2["sentiment"].notna().all()
assert submit_2["sentiment"].isin(
    ["positive", "negative"]
).all()

if os.path.exists("answer_2.pkl") or os.path.exists("answer_2.csv"):
    raise FileExistsError("기존 제출 파일이 있어 덮어쓰지 않습니다.")

shutil.copy2("answer_2_candidate.pkl", "answer_2.pkl")
submit_2.to_csv("answer_2.csv", index=False)

check_2 = pd.read_csv("answer_2.csv")

assert check_2.shape == (2000, 3)
assert check_2["id"].equals(test_2["id"])
assert check_2["review"].equals(test_2["review"])
assert np.array_equal(check_2["sentiment"], pred_test_2)

print("테스트 데이터:", test_2.shape)
print("제출 CSV:", check_2.shape)
print("CSV 예측 일치:", True)
print("모델 파일 크기:", os.path.getsize("answer_2.pkl"))
print("CSV 파일 크기:", os.path.getsize("answer_2.csv"))

print("예측 감정 분포:")
print(check_2["sentiment"].value_counts())

display(check_2.head())


모델 재불러오기 예측 일치: True
테스트 데이터: (2000, 2)
제출 CSV: (2000, 3)
CSV 예측 일치: True
모델 파일 크기: 2085169
CSV 파일 크기: 206235
예측 감정 분포:
sentiment
negative    1029
positive     971
Name: count, dtype: int64


,id,review,sentiment
0,1,로보캅 처럼 재미가 시대에 뒤 떨어짐.,negative
1,2,"칼, 도끼, 쇠뭉둥이 들고 싸우고 살인까지 하는 고딩들... 아무리 영화라지만 너무...",negative
2,3,구스 반 산트 최악의 영화임. 너무나 스타일리쉬해보이려는 노력이 처절해보일 정도. ...,negative
3,4,최고에요 자꾸 여운이 남는 영화.. 단한번도 지루할틈이 없었습니다!!,positive
4,5,다양한 상상력에한표,positive


<details><summary>힌트 1단계 (접근 방향) - 2번 문항</summary>

라벨 분포와 문장 길이를 확인합니다. 문장을 TF-IDF 로 바꾸되 **단어 단위와 글자 조각(char n-gram)** 을 비교하고, 나이브 베이즈·선형 SVM·로지스틱 회귀 중 **교차검증 정확도** 가 높은 조합으로 전체 학습하세요. 다른 컬럼(숫자·범주)이 단서이면 `ColumnTransformer` 로 함께 넣습니다.

- 이 문항의 핵심: 한국어는 조사·어미가 붙어 같은 단어도 모양이 여러 가지입니다. `TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 3))` 을 써 보세요.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 2번 문항</summary>

사용할 도구: `pandas`, `joblib`, `make_pipeline`, `CountVectorizer`, `TfidfVectorizer`, `MultinomialNB`, `LinearSVC`, `train_test_split`, `accuracy_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 2번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
import ____
from sklearn.pipeline import ____
from sklearn.feature_extraction.text import ____, ____
from sklearn.naive_bayes import ____
from sklearn.svm import ____
from sklearn.model_selection import ____
from sklearn.metrics import ____

train = pd.read_csv('02_train.csv')
test = pd.read_csv('02_test_x.csv')
TEXT, LABEL = '____', '____'
print(train[LABEL].value_counts().to_dict(), '____', round(train[TEXT].str.len().mean(), ____))


# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_2.pkl`(모델) · `answer_2.csv`(예측)

</details>

## 3. Image 문제 - 콩잎 사진으로 병해 진단(3종) (35점)

<sub>학습 목표: 콩잎 사진으로 병해 진단(3종) 모델로 테스트 정확도 0.77 이상을 달성할 수 있다 | 역량: PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | 기술: pandas, scikit-learn(LogisticRegression), Keras(TensorFlow.js), joblib | 난이도: 고급 | 예상 시간: 60분</sub>

### 문제 개요

스마트폰으로 찍은 콩잎 사진으로 **각진 잎 반점병 · 녹병 · 건강한 잎** 3종을 구분하는 이미지 분류 문제입니다. 학습 이미지가 종류당 약 240장이라 **사전학습 모델 전이학습** 이 효과적입니다.

제공된 훈련 데이터셋을 활용해 이미지 다중 분류(3종) 모델을 개발하고, 테스트 데이터셋의 'label' 값을 예측하시기 바랍니다.

### 달성 목표

"03_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 77% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `03_images.zip`: 압축을 풀면 `03_train/<클래스>/bean_*.png`(학습)와 `03_test/test_*.png`(테스트) 폴더가 생깁니다. 128×128 컬러 PNG.
- `03_train.csv`: 학습 이미지 파일명과 라벨
- `03_test_x.csv`: 테스트 이미지 파일명(라벨 제외)

**Label 정보**

- 'label' : angular_leaf_spot: 각진 잎 반점병 / bean_rust: 녹병 / healthy: 건강한 잎

**데이터 출처** — iBean 콩잎 병해 사진(HF AI-Lab-Makerere/beans · MIT, Makerere AI Lab) — 우간다 현장 촬영, 가운데 정사각형 128×128

### 데이터 예시

학습 이미지 라벨별 장수: bean_rust 240, angular_leaf_spot 240, healthy 240

| file_name | label |
|---|---|
| bean_00001.png | bean_rust |
| bean_00002.png | angular_leaf_spot |
| bean_00003.png | healthy |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 콩잎 사진으로 병해 진단(3종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_3.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('label') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_3.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'label' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'label' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 주피터라이트에는 **MobileNet v1(alpha=0.25, 224×224, ImageNet) 가중치가 미리 들어 있습니다**. `MobileNet(input_shape=(224, 224, 3), alpha=0.25, include_top=False, weights='imagenet', pooling='avg')` 로 특징만 뽑고(가중치 고정), 그 특징으로 로지스틱 회귀 같은 분류기를 학습하세요. 다른 사전학습 모델(MobileNetV2·ResNet 등)은 브라우저에서 쓸 수 없습니다.
- MobileNet 전용 `preprocess_input` 을 쓰고, 1/255 rescale 을 **함께 하지 마세요**(이중 정규화).
- 브라우저에서는 224×224 이미지를 한 번에 많이 올리면 느려집니다. **64장씩 나눠** 특징을 뽑으세요.
- 병반은 잎의 작은 무늬라 이미지를 너무 작게 줄이면 구분이 어려워집니다. 224×224 로 키워 MobileNet 에 넣으세요.
- 평가 지표는 **정확도(Accuracy)** 입니다. 테스트는 `03_test_x.csv` 의 파일명 순서대로 읽어 예측하세요.

### 채점 진행 방식 안내

- 3번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_3.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 69.3% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [16]:

import pandas as pd
import zipfile
from PIL import Image
from io import BytesIO

train_3 = pd.read_csv("03_train.csv")
test_3 = pd.read_csv("03_test_x.csv")

print("학습 데이터:", train_3.shape)
print("테스트 데이터:", test_3.shape)

display(train_3.head())

print("클래스 분포:")
print(train_3["label"].value_counts())

print("결측치:")
print(train_3.isna().sum())

print("중복 파일명:", train_3["file_name"].duplicated().sum())

with zipfile.ZipFile("03_images.zip") as z:
    image_names_3 = set(
        name for name in z.namelist()
        if name.endswith(".png")
    )

    train_paths_3 = [
        f"03_train/{label}/{name}"
        for name, label in zip(
            train_3["file_name"], train_3["label"]
        )
    ]

    test_paths_3 = [
        f"03_test/{name}"
        for name in test_3["file_name"]
    ]

    missing_train_3 = set(train_paths_3) - image_names_3
    missing_test_3 = set(test_paths_3) - image_names_3

    print("ZIP 이미지 수:", len(image_names_3))
    print("학습 이미지 누락:", len(missing_train_3))
    print("테스트 이미지 누락:", len(missing_test_3))

    print("클래스별 이미지 정보:")

    for label in sorted(train_3["label"].unique()):
        path = next(
            p for p in train_paths_3
            if p.startswith(f"03_train/{label}/")
        )

        with Image.open(BytesIO(z.read(path))) as img:
            print(label, img.size, img.mode)


학습 데이터: (720, 2)
테스트 데이터: (180, 1)


,file_name,label
0,bean_00001.png,bean_rust
1,bean_00002.png,angular_leaf_spot
2,bean_00003.png,healthy
3,bean_00004.png,healthy
4,bean_00005.png,angular_leaf_spot


클래스 분포:
label
bean_rust            240
angular_leaf_spot    240
healthy              240
Name: count, dtype: int64
결측치:
file_name    0
label        0
dtype: int64
중복 파일명: 0
ZIP 이미지 수: 900
학습 이미지 누락: 0
테스트 이미지 누락: 0
클래스별 이미지 정보:
angular_leaf_spot (128, 128) RGB
bean_rust (128, 128) RGB
healthy (128, 128) RGB


In [17]:

import numpy as np
from sklearn.model_selection import train_test_split

train_split_3, valid_split_3 = train_test_split(
    train_3,
    test_size=0.2,
    random_state=42,
    stratify=train_3["label"]
)

class_names_3 = sorted(train_3["label"].unique())

label_to_id_3 = {
    label: i for i, label in enumerate(class_names_3)
}

y_train_3 = train_split_3["label"].map(label_to_id_3).to_numpy()
y_valid_3 = valid_split_3["label"].map(label_to_id_3).to_numpy()

train_paths_split_3 = [
    f"03_train/{label}/{name}"
    for name, label in zip(
        train_split_3["file_name"], train_split_3["label"]
    )
]

valid_paths_3 = [
    f"03_train/{label}/{name}"
    for name, label in zip(
        valid_split_3["file_name"], valid_split_3["label"]
    )
]

def load_batch_3(z, paths):
    images = []

    for path in paths:
        with Image.open(BytesIO(z.read(path))) as img:
            img = img.convert("RGB").resize(
                (224, 224), Image.Resampling.BILINEAR
            )
            images.append(np.asarray(img, dtype=np.float32))

    return np.stack(images)

with zipfile.ZipFile("03_images.zip") as z:
    sample_3 = load_batch_3(z, [train_paths_split_3[0]])

sample_normalized_3 = sample_3 / 127.5 - 1.0

print("클래스 인코딩:", label_to_id_3)
print("학습 이미지:", len(train_paths_split_3))
print("검증 이미지:", len(valid_paths_3))

print("학습 클래스 분포:")
print(train_split_3["label"].value_counts())

print("검증 클래스 분포:")
print(valid_split_3["label"].value_counts())

print("이미지 배열:", sample_3.shape)
print("정규화 범위:",
      sample_normalized_3.min(),
      sample_normalized_3.max())


클래스 인코딩: {'angular_leaf_spot': 0, 'bean_rust': 1, 'healthy': 2}
학습 이미지: 576
검증 이미지: 144
학습 클래스 분포:
label
angular_leaf_spot    192
healthy              192
bean_rust            192
Name: count, dtype: int64
검증 클래스 분포:
label
angular_leaf_spot    48
bean_rust            48
healthy              48
Name: count, dtype: int64
이미지 배열: (1, 224, 224, 3)
정규화 범위: -1.0 0.96862745


In [18]:

from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input

feature_extractor_3 = MobileNet(
    input_shape=(224, 224, 3),
    alpha=0.25,
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

feature_extractor_3.trainable = False

sample_input_3 = preprocess_input(sample_3.copy())

sample_features_3 = feature_extractor_3.predict(
    sample_input_3,
    verbose=0
)

print("입력 이미지:", sample_input_3.shape)
print("전처리 범위:",
      sample_input_3.min(),
      sample_input_3.max())

print("추출된 특징:", sample_features_3.shape)
print("특징값 정상:", np.isfinite(sample_features_3).all())
print("사전학습 모델 고정:", not feature_extractor_3.trainable)


2108140/2108140 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
입력 이미지: (1, 224, 224, 3)
전처리 범위: -1.0 0.96862745
추출된 특징: (1, 256)
특징값 정상: True
사전학습 모델 고정: True


In [19]:

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score

def extract_features_3(z, paths, batch_size=64):
    features = []

    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i + batch_size]

        images = load_batch_3(z, batch_paths)
        images = preprocess_input(images)

        batch_features = feature_extractor_3.predict(
            images,
            batch_size=16,
            verbose=0
        )

        features.append(batch_features)

    return np.vstack(features)

with zipfile.ZipFile("03_images.zip") as z:
    X_train_features_3 = extract_features_3(
        z, train_paths_split_3
    )
    X_valid_features_3 = extract_features_3(
        z, valid_paths_3
    )

print("학습 특징:", X_train_features_3.shape)
print("검증 특징:", X_valid_features_3.shape)

assert np.isfinite(X_train_features_3).all()
assert np.isfinite(X_valid_features_3).all()

models_3 = {
    "LogisticRegression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    ),
    "SVC (RBF)": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=3)
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=1
    )
}

cv_3 = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

results_3 = []

for name, model in models_3.items():
    scores = cross_val_score(
        model,
        X_train_features_3,
        y_train_3,
        cv=cv_3,
        scoring="accuracy",
        n_jobs=1
    )

    model.fit(X_train_features_3, y_train_3)
    pred = model.predict(X_valid_features_3)

    results_3.append({
        "모델": name,
        "CV Accuracy": scores.mean() * 100,
        "CV 표준편차": scores.std() * 100,
        "독립 검증 Accuracy": accuracy_score(
            y_valid_3, pred
        ) * 100
    })

result_df_3 = pd.DataFrame(results_3)

display(
    result_df_3.sort_values(
        "CV Accuracy", ascending=False
    ).round(2)
)


학습 특징: (576, 256)
검증 특징: (144, 256)


,모델,CV Accuracy,CV 표준편차,독립 검증 Accuracy
1,SVC (RBF),83.68,2.53,79.86
2,RandomForest,79.17,2.25,78.47
0,LogisticRegression,77.95,3.04,79.86


In [21]:

import os
import joblib
from sklearn.base import clone

candidate_path_3 = "answer_3_candidate.pkl"

if os.path.exists(candidate_path_3):
    raise FileExistsError("기존 후보 파일이 있어 덮어쓰지 않습니다.")

X_all_features_3 = np.vstack([
    X_train_features_3,
    X_valid_features_3
])

y_all_3 = np.concatenate([
    y_train_3,
    y_valid_3
])

model_final_3 = clone(models_3["SVC (RBF)"])
model_final_3.fit(X_all_features_3, y_all_3)

weights_3 = feature_extractor_3.get_weights()

model_bundle_3 = {
    "model_type": "MobileNetV1 + SVC",
    "alpha": 0.25,
    "image_size": (224, 224),
    "feature_extractor_weights": weights_3,
    "classifier": model_final_3,
    "class_names": class_names_3
}

joblib.dump(
    model_bundle_3,
    candidate_path_3,
    compress=3
)

file_size_3 = os.path.getsize(candidate_path_3)

print("전체 학습 이미지:", len(y_all_3))
print("학습 특징:", X_all_features_3.shape)
print("분류 클래스:", model_final_3.classes_)
print("MobileNet 가중치 배열:", len(weights_3))
print("저장 파일:", candidate_path_3)
print("파일 크기:", file_size_3, "bytes")
print("저장 크기 정상:", file_size_3 > 0)


전체 학습 이미지: 720
학습 특징: (720, 256)
분류 클래스: [0 1 2]
MobileNet 가중치 배열: 135
저장 파일: answer_3_candidate.pkl
파일 크기: 1455601 bytes
저장 크기 정상: True


In [22]:

import shutil

if os.path.exists("answer_3.pkl") or os.path.exists("answer_3.csv"):
    raise FileExistsError("기존 제출 파일이 있어 덮어쓰지 않습니다.")

loaded_bundle_3 = joblib.load("answer_3_candidate.pkl")

loaded_extractor_3 = MobileNet(
    input_shape=(224, 224, 3),
    alpha=loaded_bundle_3["alpha"],
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

loaded_extractor_3.set_weights(
    loaded_bundle_3["feature_extractor_weights"]
)
loaded_extractor_3.trainable = False

restored_features_3 = loaded_extractor_3.predict(
    sample_input_3, verbose=0
)

same_features_3 = np.allclose(
    sample_features_3,
    restored_features_3,
    rtol=1e-4,
    atol=1e-4
)

same_classifier_3 = np.array_equal(
    model_final_3.predict(X_valid_features_3),
    loaded_bundle_3["classifier"].predict(X_valid_features_3)
)

print("MobileNet 특징 일치:", same_features_3)
print("SVC 예측 일치:", same_classifier_3)

assert same_features_3
assert same_classifier_3

original_extractor_3 = feature_extractor_3
feature_extractor_3 = loaded_extractor_3

try:
    with zipfile.ZipFile("03_images.zip") as z:
        X_test_features_3 = extract_features_3(
            z, test_paths_3
        )
finally:
    feature_extractor_3 = original_extractor_3

pred_ids_3 = loaded_bundle_3["classifier"].predict(
    X_test_features_3
)

pred_labels_3 = np.array(
    loaded_bundle_3["class_names"]
)[pred_ids_3.astype(int)]

submit_3 = test_3.copy()
submit_3["label"] = pred_labels_3

assert X_test_features_3.shape == (180, 256)
assert submit_3.shape == (180, 2)
assert list(submit_3.columns) == ["file_name", "label"]
assert submit_3["label"].notna().all()
assert submit_3["label"].isin(class_names_3).all()

shutil.copy2("answer_3_candidate.pkl", "answer_3.pkl")
submit_3.to_csv("answer_3.csv", index=False)

check_3 = pd.read_csv("answer_3.csv")

assert check_3["file_name"].equals(test_3["file_name"])
assert np.array_equal(check_3["label"], pred_labels_3)
assert os.path.getsize("answer_3.pkl") > 0
assert os.path.getsize("answer_3.csv") > 0

print("테스트 특징:", X_test_features_3.shape)
print("제출 CSV:", check_3.shape)
print("CSV 예측 일치:", True)
print("모델 파일 크기:", os.path.getsize("answer_3.pkl"))
print("CSV 파일 크기:", os.path.getsize("answer_3.csv"))

print("예측 클래스 분포:")
print(check_3["label"].value_counts())

display(check_3.head())


MobileNet 특징 일치: True
SVC 예측 일치: True
테스트 특징: (180, 256)
제출 CSV: (180, 2)
CSV 예측 일치: True
모델 파일 크기: 1455601
CSV 파일 크기: 4724
예측 클래스 분포:
label
bean_rust            64
angular_leaf_spot    62
healthy              54
Name: count, dtype: int64


,file_name,label
0,test_0001.png,angular_leaf_spot
1,test_0002.png,healthy
2,test_0003.png,bean_rust
3,test_0004.png,healthy
4,test_0005.png,healthy


<details><summary>힌트 1단계 (접근 방향) - 3번 문항</summary>

압축을 풀고 폴더 구조(학습: 클래스별 폴더, 테스트: 한 폴더)를 확인합니다. 이미지를 같은 크기로 읽어 0~1 로 맞추고, 검증 데이터를 떼어 **작은 CNN**(또는 사전학습 모델 특징 + 분류기)을 학습합니다. 증강·Dropout·조기 종료로 과적합을 막고, 테스트는 `03_test_x.csv` 순서대로 예측하세요.

- 이 문항의 핵심: 주피터라이트에는 **MobileNet v1(alpha=0.25, 224×224, ImageNet) 가중치가 미리 들어 있습니다**. `MobileNet(input_shape=(224, 224, 3), alpha=0.25, include_top=False, weights='imagenet', pooling='avg')` 로 특징만 뽑고(가중치 고정), 그 특징으로 로지스틱 회귀 같은 분류기를 학습하세요. 다른 사전학습 모델(MobileNetV2·ResNet 등)은 브라우저에서 쓸 수 없습니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 3번 문항</summary>

사용할 도구: `pandas`, `joblib`, `Image`, `LogisticRegression`, `train_test_split`, `accuracy_score`, `MobileNet`, `preprocess_input`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html>
- 참고 문서: <https://keras.io/api/applications/mobilenet/>

</details>

<details><summary>힌트 3단계 (코드 구조) - 3번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
import ____
import ____
from PIL import ____
from sklearn.linear_model import ____
from sklearn.model_selection import ____
from sklearn.metrics import ____
from tensorflow.keras.applications import ____
from tensorflow.keras.applications.mobilenet import ____

IMG_DIR = os.environ.get('____', '____')        # 압축을 풀 위치(브라우저판은 메모리 폴더 /tmp)
if not os.path.exists(os.path.join(IMG_DIR, '____')):
    zipfile.ZipFile('03_images.zip').extractall(IMG_DIR)
train = pd.read_csv('03_train.csv')
test = pd.read_csv('03_test_x.csv')

# ▷ 모델 구성(CNN 또는 사전학습 특징 추출)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_3.pkl`(모델) · `answer_3.csv`(예측)

</details>